# PhishGuard SHAP Analysis

This notebook explains a sample from the saved four-class Logistic Regression model. It loads the canonical feature dataset and model bundle, then selects SHAP contributions for the model's predicted class.

In [ ]:
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import train_test_split

project_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "models" / "final_model.pkl").is_file()
)
processed = project_root / "ml" / "data" / "processed"
feature_data = pd.read_csv(processed / "feature_dataset.csv")
feature_names = json.loads((processed / "feature_names.json").read_text(encoding="utf-8"))
with (project_root / "models" / "final_model.pkl").open("rb") as model_file:
    bundle = pickle.load(model_file)

model = bundle["model"]
scaler = bundle["scaler"]
X = feature_data[feature_names]
y = feature_data["label"]
X_train, X_test, _, _ = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train_scaled = scaler.transform(X_train)
sample_count = min(200, len(X_test))
X_sample = X_test.iloc[:sample_count]
X_sample_scaled = scaler.transform(X_sample)

explainer = shap.LinearExplainer(model, X_train_scaled, feature_names=feature_names)
raw_shap_values = explainer.shap_values(X_sample_scaled)
prediction = int(model.predict(X_sample_scaled[:1])[0])
class_position = list(model.classes_).index(prediction)
class_labels = {int(key): value for key, value in bundle["class_labels"].items()}
probabilities = model.predict_proba(X_sample_scaled[:1])[0]

if isinstance(raw_shap_values, list):
    class_shap_values = np.asarray(raw_shap_values[class_position])
else:
    shap_array = np.asarray(raw_shap_values)
    class_shap_values = shap_array[:, :, class_position] if shap_array.ndim == 3 else shap_array

local_values = class_shap_values[0]
print("Model:", type(model).__name__)
print("Features:", len(feature_names))
print("Predicted class:", class_labels[prediction])
print("Predicted-class probability:", probabilities[class_position])
print(pd.DataFrame({"feature": feature_names, "value": X_sample.iloc[0].values, "shap_value": local_values})
      .assign(abs_shap=lambda frame: frame["shap_value"].abs())
      .sort_values("abs_shap", ascending=False)
      .head(5)
      .to_string(index=False))

global_explanation = shap.Explanation(
    values=class_shap_values,
    data=X_sample.to_numpy(),
    feature_names=feature_names,
)
shap.plots.bar(global_explanation, max_display=10, show=False)
plt.tight_layout()
plt.savefig(processed / "shap_global_importance.png", dpi=200)
plt.close()

expected_values = np.asarray(explainer.expected_value)
base_value = float(expected_values[class_position]) if expected_values.ndim else float(expected_values)
local_explanation = shap.Explanation(
    values=local_values,
    base_values=base_value,
    data=X_sample.iloc[0].to_numpy(),
    feature_names=feature_names,
)
shap.plots.waterfall(local_explanation, max_display=10, show=False)
plt.tight_layout()
plt.savefig(processed / "shap_local_explanation.png", dpi=200)
plt.close()